In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from delta.tables import DeltaTable
from datetime import datetime
import uuid

# =============================================================================
# GENERIC AUDIT LOGGING FUNCTION
# Reusable across Bronze, Silver, and Gold layers
# =============================================================================

def log_audit_ingestion(
    pipeline_name: str,
    layer: str,  # "bronze", "silver", or "gold"
    source_name: str,
    target_table: str,
    source_type: str = "file",
    bronze_table: str = None,
    batch_id: str = None,
    run_id: str = None,
    trigger_type: str = "manual",
    attempt: int = 1,
    run_start_ts = None,
    run_end_ts = None,
    last_status: str = "SUCCESS",
    records_read: int = 0,
    records_written: int = 0,
    error_count: int = 0,
    error_message: str = None,
    watermark_col: str = None,
    last_success_watermark_value: str = None,
    current_run_high_watermark_value: str = None,
    file_checkpoint_path: str = None,
    schema_version_applied: str = None,
    producer_system: str = None,
    ingestion_user: str = None,
    notes: str = None,
    last_load_date = None
):
    """
    Generic audit logging function for Bronze, Silver, and Gold layers.
    
    Parameters:
    -----------
    pipeline_name : str (REQUIRED)
        Name of the pipeline (e.g., "HC_Bronze_Ingestion", "HC_Silver_SCD2")
    layer : str (REQUIRED)
        Layer name: "bronze", "silver", or "gold"
    source_name : str (REQUIRED)
        Name of the source (e.g., "employees", "departments")
    target_table : str (REQUIRED)
        Full table name written to (e.g., "edl_hc_datamart.silver.employees")
    source_type : str
        Type of source: "file", "table", "api", etc.
    bronze_table : str
        Bronze table name (for silver/gold layers referencing bronze)
    batch_id : str
        Unique batch identifier
    run_id : str
        Unique run identifier (auto-generated if not provided)
    trigger_type : str
        How was this triggered: "manual", "scheduled", "event"
    attempt : int
        Attempt number (for retries)
    run_start_ts : timestamp
        Start timestamp (auto-captured if not provided)
    run_end_ts : timestamp
        End timestamp (auto-captured if not provided)
    last_status : str
        Status: "SUCCESS", "FAILED", "RUNNING", "PARTIAL"
    records_read : int
        Number of records read from source
    records_written : int
        Number of records written to target
    error_count : int
        Number of errors encountered
    error_message : str
        Error details if failed
    watermark_col : str
        Column used for incremental watermark
    last_success_watermark_value : str
        Last successful watermark value
    current_run_high_watermark_value : str
        Current run's high watermark value
    file_checkpoint_path : str
        Path to checkpoint file for incremental loads
    schema_version_applied : str
        Schema version applied
    producer_system : str
        Source system name
    ingestion_user : str
        User who triggered the ingestion
    notes : str
        Additional notes
    last_load_date : date
        Load date parameter
    
    Returns:
    --------
    str : run_id of the logged audit record
    
    Example Usage:
    --------------
    # Bronze layer
    log_audit_ingestion(
        pipeline_name="HC_Bronze_Ingestion",
        layer="bronze",
        source_name="employees",
        target_table="edl_hc_datamart.bronze.employees",
        records_read=1500,
        records_written=1500,
        last_load_date="2026-02-16"
    )
    
    # Silver layer
    log_audit_ingestion(
        pipeline_name="HC_Silver_SCD2",
        layer="silver",
        source_name="employees",
        target_table="edl_hc_datamart.silver.employees",
        source_type="table",
        bronze_table="edl_hc_datamart.bronze.employees",
        records_read=1500,
        records_written=3200  # includes historical versions
    )
    """
    
    # Auto-generate run_id if not provided
    if run_id is None:
        run_id = str(uuid.uuid4())
    
    # Auto-capture timestamps if not provided
    current_ts = datetime.now()
    if run_start_ts is None:
        run_start_ts = current_ts
    if run_end_ts is None:
        run_end_ts = current_ts
    
    # Calculate duration in milliseconds
    duration_ms = None
    try:
        if isinstance(run_start_ts, datetime) and isinstance(run_end_ts, datetime):
            duration_ms = int((run_end_ts - run_start_ts).total_seconds() * 1000)
    except:
        pass  # Will be calculated in SQL if timestamps are provided
    
    # Prepare audit record
    audit_data = [
        (
            pipeline_name,
            layer,  # NEW COLUMN
            source_type,
            source_name,
            bronze_table,
            target_table,  # NEW COLUMN
            batch_id,
            run_id,
            trigger_type,
            attempt,
            run_start_ts,
            run_end_ts,
            duration_ms,
            last_status,
            records_read,
            records_written,
            error_count,
            error_message,
            watermark_col,
            last_success_watermark_value,
            current_run_high_watermark_value,
            file_checkpoint_path,
            schema_version_applied,
            producer_system,
            ingestion_user,
            notes,
            last_load_date,
            current_ts,  # created_at
            current_ts   # updated_at
        )
    ]
    
    # Define explicit schema with proper data types
    audit_schema = StructType([
        StructField("pipeline_name", StringType(), True),
        StructField("layer", StringType(), True),
        StructField("source_type", StringType(), True),
        StructField("source_name", StringType(), True),
        StructField("bronze_table", StringType(), True),
        StructField("target_table", StringType(), True),
        StructField("batch_id", StringType(), True),
        StructField("run_id", StringType(), True),
        StructField("trigger_type", StringType(), True),
        StructField("attempt", IntegerType(), True),
        StructField("run_start_ts", TimestampType(), True),
        StructField("run_end_ts", TimestampType(), True),
        StructField("duration_ms", IntegerType(), True),
        StructField("last_status", StringType(), True),
        StructField("records_read", IntegerType(), True),
        StructField("records_written", IntegerType(), True),
        StructField("error_count", IntegerType(), True),
        StructField("error_message", StringType(), True),
        StructField("watermark_col", StringType(), True),
        StructField("last_success_watermark_value", StringType(), True),
        StructField("current_run_high_watermark_value", StringType(), True),
        StructField("file_checkpoint_path", StringType(), True),
        StructField("schema_version_applied", StringType(), True),
        StructField("producer_system", StringType(), True),
        StructField("ingestion_user", StringType(), True),
        StructField("notes", StringType(), True),
        StructField("last_load_date", StringType(), True),
        StructField("created_at", TimestampType(), True),
        StructField("updated_at", TimestampType(), True)
    ])
    
    # Create DataFrame
    audit_df = spark.createDataFrame(audit_data, audit_schema)
    
    # Calculate duration_ms in SQL if not already calculated
    if duration_ms is None:
        audit_df = audit_df.withColumn(
            "duration_ms",
            F.when(
                (F.col("run_start_ts").isNotNull()) & (F.col("run_end_ts").isNotNull()),
                (F.unix_timestamp("run_end_ts") - F.unix_timestamp("run_start_ts")) * 1000
            ).otherwise(F.lit(None))
        )
    
    # Write to audit table
    audit_table = "edl_hc_datamart.audit.audit_ingestion"
    
    # Check if table exists
    if spark.catalog.tableExists(audit_table):
        # Use merge to update existing records or insert new ones
        target = DeltaTable.forName(spark, audit_table)
        
        target.alias("tgt").merge(
            audit_df.alias("src"),
            "tgt.pipeline_name = src.pipeline_name AND tgt.source_name = src.source_name AND tgt.run_id = src.run_id"
        ).whenMatchedUpdate(set={
            "layer": "src.layer",
            "source_type": "src.source_type",
            "bronze_table": "src.bronze_table",
            "target_table": "src.target_table",
            "batch_id": "src.batch_id",
            "trigger_type": "src.trigger_type",
            "attempt": "src.attempt",
            "run_start_ts": "src.run_start_ts",
            "run_end_ts": "src.run_end_ts",
            "duration_ms": "src.duration_ms",
            "last_status": "src.last_status",
            "records_read": "src.records_read",
            "records_written": "src.records_written",
            "error_count": "src.error_count",
            "error_message": "src.error_message",
            "watermark_col": "src.watermark_col",
            "last_success_watermark_value": "src.last_success_watermark_value",
            "current_run_high_watermark_value": "src.current_run_high_watermark_value",
            "file_checkpoint_path": "src.file_checkpoint_path",
            "schema_version_applied": "src.schema_version_applied",
            "producer_system": "src.producer_system",
            "ingestion_user": "src.ingestion_user",
            "notes": "src.notes",
            "last_load_date": "src.last_load_date",
            "updated_at": "src.updated_at"
        }).whenNotMatchedInsertAll().execute()
        
        print(f"✓ Audit log updated: {audit_table}")
    else:
        # First time - create the table
        audit_df.write.format("delta").mode("append").saveAsTable(audit_table)
        print(f"✓ Audit log created and logged: {audit_table}")
    
    print(f"Run ID: {run_id} | Status: {last_status} | Records: {records_read} → {records_written}")
    return run_id


# =============================================================================
# HELPER FUNCTION: Context-based logging with try/except wrapper
# =============================================================================

def audit_pipeline_run(layer: str, source_name: str, target_table: str, pipeline_func, **kwargs):
    """
    Wrapper function that executes pipeline logic and logs audit automatically.
    
    Parameters:
    -----------
    layer : str
        "bronze", "silver", or "gold"
    source_name : str
        Source name (e.g., "employees")
    target_table : str
        Target table full name
    pipeline_func : callable
        Function to execute (should return records_read, records_written)
    **kwargs : additional audit parameters
    
    Example:
    --------
    def process_employees():
        # ... processing logic ...
        return 1500, 1500  # records_read, records_written
    
    audit_pipeline_run(
        layer="bronze",
        source_name="employees",
        target_table="edl_hc_datamart.bronze.employees",
        pipeline_func=process_employees,
        pipeline_name="HC_Bronze_Ingestion"
    )
    """
    start_ts = datetime.now()
    status = "RUNNING"
    error_msg = None
    records_read = 0
    records_written = 0
    
    try:
        # Execute the pipeline function
        result = pipeline_func()
        
        # Extract metrics if returned
        if isinstance(result, tuple) and len(result) == 2:
            records_read, records_written = result
        
        status = "SUCCESS"
    
    except Exception as e:
        status = "FAILED"
        error_msg = str(e)
        print(f"✗ Pipeline failed: {error_msg}")
        raise  # Re-raise to not hide the error
    
    finally:
        end_ts = datetime.now()
        
        # Log audit regardless of success/failure
        log_audit_ingestion(
            layer=layer,
            source_name=source_name,
            target_table=target_table,
            run_start_ts=start_ts,
            run_end_ts=end_ts,
            last_status=status,
            records_read=records_read,
            records_written=records_written,
            error_message=error_msg,
            error_count=1 if error_msg else 0,
            **kwargs
        )

print("✓ Audit logging functions loaded successfully!")
print("  - log_audit_ingestion(): Direct logging function")
print("  - audit_pipeline_run(): Wrapper with automatic try/except")

In [0]:
%sql
-- Add new columns to existing audit table
-- Run this once to enhance your audit table schema

-- ALTER TABLE edl_hc_datamart.audit.audit_ingestion 
-- ADD COLUMNS (
--   layer STRING COMMENT 'Pipeline layer: bronze, silver, or gold',
--   target_table STRING COMMENT 'Full name of target table written to'
-- );

-- -- Verify the columns were added
-- DESCRIBE edl_hc_datamart.audit.audit_ingestion;

In [0]:
# =============================================================================
# EXAMPLE: How to integrate audit logging into your Bronze pipeline
# =============================================================================

# This is a template showing how to add audit logging to your existing bronze ingestion
# You would integrate this into Cell 2 of this notebook

"""
INTEGRATION APPROACH 1: Add audit logging after each table write

In your main ingestion loop (Cell 2), after the line:
    df_enriched.write.mode("overwrite").format("delta").saveAsTable(full_table)

Add:
    # Count records for audit
    records_written = df_enriched.count()
    records_read = df.count()  # before enrichment
    
    # Log audit
    log_audit_ingestion(
        pipeline_name="HC_Bronze_Ingestion",
        layer="bronze",
        source_name=source_name,
        target_table=full_table,
        source_type=source_type,
        batch_id=md.get('batch_id'),
        run_id=md.get('run_id'),
        trigger_type="manual",  # or "scheduled" if job-triggered
        records_read=records_read,
        records_written=records_written,
        schema_version_applied=md.get('schema_version'),
        producer_system=md.get('producer_system'),
        ingestion_user=md.get('ingestion_user'),
        last_load_date=LOAD_DATE
    )
"""

# =============================================================================
# INTEGRATION APPROACH 2: Wrapper function (cleaner but requires refactoring)
# =============================================================================

"""
You can wrap your processing logic in the audit_pipeline_run function:

def process_source(source_name, landed_path, full_table, options, fmt):
    '''Process a single source and return metrics'''
    
    # Read data
    reader = spark.read
    for k, v in options.items():
        reader = reader.option(k, v)
    df = reader.format(fmt).load(landed_path)
    records_read = df.count()
    
    # ... all your transformation logic ...
    df_enriched = df.withColumn(...)
    
    # Write
    df_enriched.write.mode("overwrite").format("delta").saveAsTable(full_table)
    records_written = df_enriched.count()
    
    return records_read, records_written

# Then call with audit wrapper:
for m in meta_df.collect():
    # ... setup logic ...
    
    audit_pipeline_run(
        layer="bronze",
        source_name=source_name,
        target_table=full_table,
        pipeline_func=lambda: process_source(source_name, landed_path, full_table, options, fmt),
        pipeline_name="HC_Bronze_Ingestion",
        batch_id=md.get('batch_id'),
        run_id=md.get('run_id')
    )
"""

# =============================================================================
# QUICK TEST: Log a sample audit entry
# COMMENTED OUT - Uncomment to test manually
# =============================================================================

# print("\n" + "="*80)
# print("TESTING AUDIT LOGGING")
# print("="*80)

# Test with sample data
# test_run_id = log_audit_ingestion(
#     pipeline_name="HC_Bronze_Ingestion",
#     layer="bronze",
#     source_name="employees_test",
#     target_table="edl_hc_datamart.bronze.employees",
#     source_type="file",
#     trigger_type="manual",
#     records_read=1500,
#     records_written=1500,
#     last_status="SUCCESS",
#     last_load_date="2026-02-16",
#     notes="Test audit log entry from generic notebook"
# )

# print(f"\n✓ Test audit entry created with run_id: {test_run_id}")
# print("\nQuery your audit table:")
# print("SELECT * FROM edl_hc_datamart.audit.audit_ingestion WHERE pipeline_name = 'HC_Bronze_Ingestion' ORDER BY created_at DESC LIMIT 5")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, LongType
from delta.tables import DeltaTable
from datetime import datetime
import uuid
import traceback
import sys

# =============================================================================
# ERROR LOGGING FUNCTION
# Captures detailed error information for pipeline failures
# =============================================================================

def log_pipeline_error(
    pipeline_name: str,
    layer: str,
    source_name: str,
    target_table: str,
    error_message: str,
    # Job/Task Context
    job_id: str = None,
    job_run_id: str = None,
    task_run_id: str = None,
    task_key: str = None,
    run_as: str = None,
    launched_by: str = None,
    # Timing
    job_started_at = None,
    job_ended_at = None,
    duration_seconds: int = None,
    # Error Details
    error_type: str = None,
    error_code: str = None,
    error_stacktrace: str = None,
    failed_statement: str = None,
    exception_obj: Exception = None,
    # Compute Context
    cluster_id: str = None,
    warehouse_id: str = None,
    notebook_path: str = None,
    cell_guid: str = None,
    # Retry Context
    attempt_number: int = 1,
    is_retry: bool = False,
    parent_run_id: str = None,
    # Linking
    audit_run_id: str = None,
    batch_id: str = None,
    # Additional Context
    notes: str = None,
    error_id: str = None
):
    """
    Log detailed error information for pipeline failures.
    
    Parameters:
    -----------
    pipeline_name : str (REQUIRED)
        Name of the pipeline that failed
    layer : str (REQUIRED)
        Layer: "bronze", "silver", or "gold"
    source_name : str (REQUIRED)
        Source name being processed
    target_table : str (REQUIRED)
        Target table that was being written to
    error_message : str (REQUIRED)
        Main error message
    
    Job/Task Context:
    -----------------
    job_id : str
        Databricks Job ID
    job_run_id : str
        Specific job run ID
    task_run_id : str
        Task run ID within the job
    task_key : str
        Task key/name in the job
    run_as : str
        User email who triggered the run
    launched_by : str
        How was it launched: "manual", "scheduler", "retry scheduler", "api", etc.
    
    Timing:
    -------
    job_started_at : timestamp
        When the job started
    job_ended_at : timestamp
        When the job ended/failed
    duration_seconds : int
        Duration in seconds
    
    Error Details:
    --------------
    error_type : str
        Category: "SCHEMA_MISMATCH", "DATA_QUALITY", "PERMISSION_DENIED", 
        "TIMEOUT", "RESOURCE_EXHAUSTED", "CONNECTION_ERROR", etc.
    error_code : str
        Specific error code (e.g., "DELTA_METADATA_MISMATCH")
    error_stacktrace : str
        Full stacktrace
    failed_statement : str
        SQL statement or code that failed
    exception_obj : Exception
        Python exception object (will extract stacktrace automatically)
    
    Compute Context:
    ----------------
    cluster_id : str
        Cluster ID where error occurred
    warehouse_id : str
        SQL Warehouse ID if applicable
    notebook_path : str
        Path to notebook
    cell_guid : str
        Cell GUID if error in notebook
    
    Retry Context:
    --------------
    attempt_number : int
        Which attempt number (1 for first try)
    is_retry : bool
        Whether this is a retry attempt
    parent_run_id : str
        If retry, the original run_id
    
    Linking:
    --------
    audit_run_id : str
        Links to audit_ingestion.run_id
    batch_id : str
        Batch identifier
    
    Returns:
    --------
    str : error_id of the logged error
    
    Example Usage:
    --------------
    try:
        # ... pipeline code ...
        df.write.saveAsTable(target_table)
    except Exception as e:
        log_pipeline_error(
            pipeline_name="HC_Bronze_Ingestion",
            layer="bronze",
            source_name="employees",
            target_table="edl_hc_datamart.bronze.employees",
            error_message=str(e),
            exception_obj=e,
            error_type="SCHEMA_MISMATCH",
            job_id=dbutils.jobs.taskValues.get(taskKey="job_id"),
            run_as=spark.sql("SELECT current_user()").collect()[0][0],
            audit_run_id=run_id
        )
        raise
    """
    
    # Auto-generate error_id if not provided
    if error_id is None:
        error_id = str(uuid.uuid4())
    
    # Auto-extract stacktrace from exception if provided
    if exception_obj is not None and error_stacktrace is None:
        error_stacktrace = ''.join(traceback.format_exception(
            type(exception_obj), 
            exception_obj, 
            exception_obj.__traceback__
        ))
    
    # Auto-extract error_code from error_message if it matches pattern [ERROR_CODE]
    if error_code is None and error_message:
        import re
        code_match = re.search(r'\[([A-Z_]+)\]', error_message)
        if code_match:
            error_code = code_match.group(1)
    
    # Calculate duration if timestamps provided
    if duration_seconds is None and job_started_at and job_ended_at:
        try:
            if isinstance(job_started_at, datetime) and isinstance(job_ended_at, datetime):
                duration_seconds = int((job_ended_at - job_started_at).total_seconds())
        except:
            pass
    
    # Get current user if run_as not provided
    if run_as is None:
        try:
            run_as = spark.sql("SELECT current_user()").collect()[0][0]
        except:
            pass
    
    # Current timestamp
    current_ts = datetime.now()
    error_timestamp = job_ended_at if job_ended_at else current_ts
    
    # Prepare error record
    error_data = [
        (
            error_id,
            audit_run_id,
            batch_id,
            pipeline_name,
            layer,
            source_name,
            target_table,
            # Job Context
            job_id,
            job_run_id,
            task_run_id,
            task_key,
            run_as,
            launched_by,
            # Timing
            error_timestamp,
            job_started_at,
            job_ended_at,
            duration_seconds,
            # Error Details
            "FAILED",  # error_status
            error_type,
            error_code,
            error_message,
            error_stacktrace,
            failed_statement,
            # Compute Context
            cluster_id,
            warehouse_id,
            notebook_path,
            cell_guid,
            # Retry Context
            attempt_number,
            is_retry,
            parent_run_id,
            # Resolution (defaults)
            "OPEN",  # resolution_status
            None,    # resolution_notes
            None,    # resolved_by
            None,    # resolved_at
            # Additional
            notes,
            # Audit
            current_ts,  # created_at
            current_ts   # updated_at
        )
    ]
    
    # Define schema
    error_schema = StructType([
        StructField("error_id", StringType(), True),
        StructField("audit_run_id", StringType(), True),
        StructField("batch_id", StringType(), True),
        StructField("pipeline_name", StringType(), True),
        StructField("layer", StringType(), True),
        StructField("source_name", StringType(), True),
        StructField("target_table", StringType(), True),
        # Job Context
        StructField("job_id", StringType(), True),
        StructField("job_run_id", StringType(), True),
        StructField("task_run_id", StringType(), True),
        StructField("task_key", StringType(), True),
        StructField("run_as", StringType(), True),
        StructField("launched_by", StringType(), True),
        # Timing
        StructField("error_timestamp", TimestampType(), True),
        StructField("job_started_at", TimestampType(), True),
        StructField("job_ended_at", TimestampType(), True),
        StructField("duration_seconds", IntegerType(), True),
        # Error Details
        StructField("error_status", StringType(), True),
        StructField("error_type", StringType(), True),
        StructField("error_code", StringType(), True),
        StructField("error_message", StringType(), True),
        StructField("error_stacktrace", StringType(), True),
        StructField("failed_statement", StringType(), True),
        # Compute Context
        StructField("cluster_id", StringType(), True),
        StructField("warehouse_id", StringType(), True),
        StructField("notebook_path", StringType(), True),
        StructField("cell_guid", StringType(), True),
        # Retry Context
        StructField("attempt_number", IntegerType(), True),
        StructField("is_retry", StringType(), True),
        StructField("parent_run_id", StringType(), True),
        # Resolution
        StructField("resolution_status", StringType(), True),
        StructField("resolution_notes", StringType(), True),
        StructField("resolved_by", StringType(), True),
        StructField("resolved_at", TimestampType(), True),
        # Additional
        StructField("notes", StringType(), True),
        # Audit
        StructField("created_at", TimestampType(), True),
        StructField("updated_at", TimestampType(), True)
    ])
    
    # Create DataFrame
    error_df = spark.createDataFrame(error_data, error_schema)
    
    # Convert boolean to string
    error_df = error_df.withColumn(
        "is_retry",
        F.when(F.col("is_retry") == "True", "true")
         .when(F.col("is_retry") == "False", "false")
         .otherwise(F.col("is_retry"))
    )
    
    # Write to error table
    error_table = "edl_hc_datamart.audit.error_log"
    
    try:
        if spark.catalog.tableExists(error_table):
            # Append to existing table
            error_df.write.format("delta").mode("append").saveAsTable(error_table)
            print(f"✓ Error logged: {error_table}")
        else:
            # First time - create the table
            error_df.write.format("delta").mode("append").saveAsTable(error_table)
            print(f"✓ Error log table created: {error_table}")
        
        print(f"Error ID: {error_id} | Type: {error_type} | Code: {error_code}")
        print(f"Message: {error_message[:100]}..." if len(error_message) > 100 else f"Message: {error_message}")
        
    except Exception as log_error:
        print(f"✗ Failed to write to error log: {log_error}")
        # Don't raise - we don't want error logging to break the main pipeline
    
    return error_id


# =============================================================================
# ENHANCED AUDIT WRAPPER WITH ERROR LOGGING
# =============================================================================

def audit_pipeline_run_with_error_logging(
    layer: str,
    source_name: str,
    target_table: str,
    pipeline_func,
    # Job context (optional)
    job_id: str = None,
    job_run_id: str = None,
    task_run_id: str = None,
    task_key: str = None,
    cluster_id: str = None,
    notebook_path: str = None,
    **kwargs
):
    """
    Enhanced wrapper that logs both audit and errors.
    
    Example:
    --------
    def process_employees():
        df = spark.read.table("bronze.employees")
        # ... transformations ...
        df.write.saveAsTable("silver.employees")
        return df.count(), df.count()
    
    audit_pipeline_run_with_error_logging(
        layer="silver",
        source_name="employees",
        target_table="edl_hc_datamart.silver.employees",
        pipeline_func=process_employees,
        pipeline_name="HC_Silver_SCD2",
        job_id="721817289208592",
        job_run_id="897422776138544"
    )
    """
    start_ts = datetime.now()
    status = "RUNNING"
    error_msg = None
    records_read = 0
    records_written = 0
    run_id = str(uuid.uuid4())
    
    try:
        # Execute the pipeline function
        result = pipeline_func()
        
        # Extract metrics if returned
        if isinstance(result, tuple) and len(result) == 2:
            records_read, records_written = result
        
        status = "SUCCESS"
    
    except Exception as e:
        status = "FAILED"
        error_msg = str(e)
        print(f"✗ Pipeline failed: {error_msg}")
        
        # Extract error type from exception class name
        error_type = type(e).__name__.upper()
        
        # Log detailed error
        try:
            log_pipeline_error(
                pipeline_name=kwargs.get('pipeline_name', 'UNKNOWN'),
                layer=layer,
                source_name=source_name,
                target_table=target_table,
                error_message=error_msg,
                exception_obj=e,
                error_type=error_type,
                job_id=job_id,
                job_run_id=job_run_id,
                task_run_id=task_run_id,
                task_key=task_key,
                cluster_id=cluster_id,
                notebook_path=notebook_path,
                job_started_at=start_ts,
                job_ended_at=datetime.now(),
                audit_run_id=run_id,
                launched_by=kwargs.get('trigger_type', 'manual')
            )
        except Exception as log_err:
            print(f"Warning: Could not log error details: {log_err}")
        
        raise  # Re-raise the original exception
    
    finally:
        end_ts = datetime.now()
        
        # Log audit regardless of success/failure
        log_audit_ingestion(
            layer=layer,
            source_name=source_name,
            target_table=target_table,
            run_id=run_id,
            run_start_ts=start_ts,
            run_end_ts=end_ts,
            last_status=status,
            records_read=records_read,
            records_written=records_written,
            error_message=error_msg,
            error_count=1 if error_msg else 0,
            **kwargs
        )

print("✓ Error logging functions loaded successfully!")
print("  - log_pipeline_error(): Direct error logging function")
print("  - audit_pipeline_run_with_error_logging(): Enhanced wrapper with audit + error logging")

In [0]:
%sql
-- =============================================================================
-- ERROR LOG TABLE SCHEMA
-- Run this to manually create the error_log table (optional - auto-created on first error)
-- =============================================================================


CREATE TABLE IF NOT EXISTS edl_hc_datamart.audit.error_log (
  -- Identifiers
  error_id STRING COMMENT 'Unique error identifier (UUID)',
  audit_run_id STRING COMMENT 'Links to audit_ingestion.run_id',
  batch_id STRING COMMENT 'Batch identifier',
  
  -- Pipeline Context
  pipeline_name STRING COMMENT 'Name of the failed pipeline',
  layer STRING COMMENT 'Pipeline layer: bronze, silver, gold',
  source_name STRING COMMENT 'Source being processed',
  target_table STRING COMMENT 'Target table being written to',
  
  -- Job Context
  job_id STRING COMMENT 'Databricks Job ID',
  job_run_id STRING COMMENT 'Specific job run ID',
  task_run_id STRING COMMENT 'Task run ID within job',
  task_key STRING COMMENT 'Task key/name',
  run_as STRING COMMENT 'User email who triggered the run',
  launched_by STRING COMMENT 'How launched: manual, scheduler, retry scheduler, api',
  
  -- Timing
  error_timestamp TIMESTAMP COMMENT 'When error occurred',
  job_started_at TIMESTAMP COMMENT 'Job start time',
  job_ended_at TIMESTAMP COMMENT 'Job end/fail time',
  duration_seconds INT COMMENT 'Duration in seconds',
  
  -- Error Details
  error_status STRING COMMENT 'Status: FAILED, TIMEOUT, CANCELLED',
  error_type STRING COMMENT 'Error category: SCHEMA_MISMATCH, DATA_QUALITY, PERMISSION_DENIED, etc',
  error_code STRING COMMENT 'Specific error code (e.g., DELTA_METADATA_MISMATCH)',
  error_message STRING COMMENT 'Main error message',
  error_stacktrace STRING COMMENT 'Full stacktrace',
  failed_statement STRING COMMENT 'SQL/code statement that failed',
  
  -- Compute Context
  cluster_id STRING COMMENT 'Cluster ID where error occurred',
  warehouse_id STRING COMMENT 'SQL Warehouse ID if applicable',
  notebook_path STRING COMMENT 'Notebook path',
  cell_guid STRING COMMENT 'Cell GUID if notebook error',
  
  -- Retry Context
  attempt_number INT COMMENT 'Attempt number (1 for first try)',
  is_retry STRING COMMENT 'Whether this is a retry: true/false',
  parent_run_id STRING COMMENT 'Original run_id if this is a retry',
  
  -- Resolution Tracking
  resolution_status STRING COMMENT 'OPEN, INVESTIGATING, RESOLVED, IGNORED',
  resolution_notes STRING COMMENT 'Notes about resolution',
  resolved_by STRING COMMENT 'User who resolved',
  resolved_at TIMESTAMP COMMENT 'When resolved',
  
  -- Additional Context
  notes STRING COMMENT 'Additional notes',
  
  -- Audit timestamps
  created_at TIMESTAMP COMMENT 'Record creation time',
  updated_at TIMESTAMP COMMENT 'Last update time'
)
USING DELTA
COMMENT 'Error log for pipeline failures - captures detailed error context for debugging';
-- PARTITIONED BY (DATE(error_timestamp));

-- Add comments for key columns
ALTER TABLE edl_hc_datamart.audit.error_log 
ALTER COLUMN error_id COMMENT 'Primary key - unique error identifier';

ALTER TABLE edl_hc_datamart.audit.error_log 
ALTER COLUMN audit_run_id COMMENT 'Foreign key to audit_ingestion.run_id';

SELECT 'Error log table created successfully' as status;


-- =============================================================================
-- USEFUL QUERIES FOR ERROR ANALYSIS
-- =============================================================================

-- View recent errors
-- SELECT 
--   error_timestamp,
--   pipeline_name,
--   layer,
--   source_name,
--   error_type,
--   error_code,
--   LEFT(error_message, 100) as error_preview,
--   resolution_status,
--   duration_seconds
-- FROM edl_hc_datamart.audit.error_log
-- WHERE error_timestamp >= CURRENT_DATE() - INTERVAL 7 DAYS
-- ORDER BY error_timestamp DESC
-- LIMIT 20;

-- Error summary by type
-- SELECT 
--   error_type,
--   error_code,
--   COUNT(*) as error_count,
--   COUNT(DISTINCT source_name) as affected_sources,
--   MAX(error_timestamp) as last_occurrence,
--   SUM(CASE WHEN resolution_status = 'RESOLVED' THEN 1 ELSE 0 END) as resolved_count
-- FROM edl_hc_datamart.audit.error_log
-- WHERE error_timestamp >= CURRENT_DATE() - INTERVAL 30 DAYS
-- GROUP BY error_type, error_code
-- ORDER BY error_count DESC;

-- Join with audit table for full context
-- SELECT 
--   e.error_timestamp,
--   e.pipeline_name,
--   e.layer,
--   e.source_name,
--   e.error_type,
--   e.error_message,
--   a.records_read,
--   a.records_written,
--   a.run_start_ts,
--   a.run_end_ts
-- FROM edl_hc_datamart.audit.error_log e
-- LEFT JOIN edl_hc_datamart.audit.audit_ingestion a
--   ON e.audit_run_id = a.run_id
-- WHERE e.error_timestamp >= CURRENT_DATE() - INTERVAL 7 DAYS
-- ORDER BY e.error_timestamp DESC;

In [0]:
# =============================================================================
# INTEGRATION EXAMPLES: How to use error logging in your pipelines
# =============================================================================

print("\n" + "="*80)
print("ERROR LOGGING INTEGRATION EXAMPLES")
print("="*80)

# -----------------------------------------------------------------------------
# EXAMPLE 1: Basic try/except with error logging
# -----------------------------------------------------------------------------
print("\nExample 1: Basic Error Logging in Bronze Ingestion")
print("-" * 80)

"""
def process_bronze_source():
    try:
        # Your processing logic
        df = spark.read.csv(source_path)
        df.write.mode("overwrite").saveAsTable(target_table)
        
    except Exception as e:
        # Log the error with context
        log_pipeline_error(
            pipeline_name="HC_Bronze_Ingestion",
            layer="bronze",
            source_name="employees",
            target_table="edl_hc_datamart.bronze.employees",
            error_message=str(e),
            exception_obj=e,  # Automatically extracts stacktrace
            error_type="READ_ERROR",  # Categorize the error
            notebook_path=dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get(),
            launched_by="scheduled"
        )
        raise  # Re-raise to fail the job
"""

# -----------------------------------------------------------------------------
# EXAMPLE 2: Enhanced wrapper (recommended)
# -----------------------------------------------------------------------------
print("\nExample 2: Using Enhanced Wrapper (Recommended)")
print("-" * 80)

"""
def process_silver_employees():
    '''Your silver layer processing logic'''
    source_df = spark.table("edl_hc_datamart.bronze.employees")
    
    # Apply SCD2 logic
    # ... transformations ...
    
    result_df.write.mode("overwrite").saveAsTable("edl_hc_datamart.silver.employees")
    
    return source_df.count(), result_df.count()

# Wrap with enhanced logging - captures both audit and errors automatically
audit_pipeline_run_with_error_logging(
    layer="silver",
    source_name="employees",
    target_table="edl_hc_datamart.silver.employees",
    pipeline_func=process_silver_employees,
    pipeline_name="HC_Silver_SCD2",
    job_id=spark.conf.get("spark.databricks.job.id", None),
    job_run_id=spark.conf.get("spark.databricks.job.runId", None),
    notebook_path=dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
"""

# -----------------------------------------------------------------------------
# EXAMPLE 3: Capturing job context from Databricks Jobs
# -----------------------------------------------------------------------------
print("\nExample 3: Capturing Job Context (Run from Databricks Job)")
print("-" * 80)

"""
# At the start of your notebook, capture job context:
try:
    job_context = {
        'job_id': spark.conf.get("spark.databricks.job.id"),
        'job_run_id': spark.conf.get("spark.databricks.job.runId"),
        'cluster_id': spark.conf.get("spark.databricks.clusterUsageTags.clusterId"),
        'notebook_path': dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get(),
        'run_as': spark.sql("SELECT current_user()").collect()[0][0]
    }
except:
    job_context = {}  # Running interactively

# Then use in error logging:
try:
    # ... your pipeline code ...
    pass
except Exception as e:
    log_pipeline_error(
        pipeline_name="HC_Bronze_Ingestion",
        layer="bronze",
        source_name="employees",
        target_table="edl_hc_datamart.bronze.employees",
        error_message=str(e),
        exception_obj=e,
        **job_context  # Unpack all job context
    )
    raise
"""

# -----------------------------------------------------------------------------
# EXAMPLE 4: Schema mismatch error (like your example)
# -----------------------------------------------------------------------------
print("\nExample 4: Handling Schema Mismatch Errors")
print("-" * 80)

"""
try:
    df.write.mode("append").saveAsTable(target_table)
except Exception as e:
    error_msg = str(e)
    
    # Detect error type
    if "DELTA_METADATA_MISMATCH" in error_msg:
        error_type = "SCHEMA_MISMATCH"
    elif "PERMISSION_DENIED" in error_msg:
        error_type = "PERMISSION_DENIED"
    elif "TABLE_OR_VIEW_NOT_FOUND" in error_msg:
        error_type = "TABLE_NOT_FOUND"
    else:
        error_type = "UNKNOWN"
    
    log_pipeline_error(
        pipeline_name="HC_Silver_SCD2",
        layer="silver",
        source_name="employees",
        target_table="edl_hc_datamart.silver.employees",
        error_message=error_msg,
        exception_obj=e,
        error_type=error_type,
        job_id="721817289208592",
        job_run_id="897422776138544",
        task_run_id="696190027787978",
        run_as="junaid20950@gmail.com",
        launched_by="retry scheduler",
        job_started_at=datetime(2026, 6, 5, 10, 28),
        job_ended_at=datetime(2026, 6, 5, 10, 30),
        notes="Schema changed in source without notification"
    )
    raise
"""

# -----------------------------------------------------------------------------
# EXAMPLE 5: Mark errors as resolved
# -----------------------------------------------------------------------------
print("\nExample 5: Updating Error Resolution Status")
print("-" * 80)

"""
# After fixing an issue, update the error log:
from delta.tables import DeltaTable

error_table = DeltaTable.forName(spark, "edl_hc_datamart.audit.error_log")

error_table.update(
    condition="error_id = 'abc-123-def'" +
              " AND resolution_status = 'OPEN'",
    set={
        "resolution_status": "'RESOLVED'",
        "resolution_notes": "'Fixed schema mismatch by updating bronze table schema'",
        "resolved_by": "current_user()",
        "resolved_at": "current_timestamp()",
        "updated_at": "current_timestamp()"
    }
)

print("✓ Error marked as resolved")
"""

print("\n" + "="*80)
print("✓ Integration examples ready!")
print("  Copy these patterns into your pipeline notebooks")
print("="*80)